# From Extraction to Retrieval: the Graph in Neo4j

**Loading, Cypher, the neo4j-graphrag retrievers, and NVL visualization.**

[Notebook 01](01_gliner25_knowledge_graphs.ipynb) built two knowledge graphs with GLiNER2.5 — a business-news
graph and an agent-memory graph — and left them as in-memory objects and a pile of JSON. That is where most
extraction demos stop, and it is exactly halfway.

This notebook takes the business-news graph the rest of the way:

1. **Load** it into Neo4j, idempotently, with the ontology's types becoming node labels and relationship types.
2. **Query** it with Cypher — the multi-hop questions that motivated building a graph at all.
3. **Retrieve** from it with every applicable retriever in
   [`neo4j-graphrag`](https://neo4j.com/docs/neo4j-graphrag-python/current/), and understand what each one is
   actually for.
4. **Visualize** it with [NVL](https://neo4j.com/docs/nvl/current/) through the `neo4j-viz` Python package.

### What you need

A Neo4j 5.26+ instance. Nothing else — **no API key**. The embeddings are a local 90 MB MiniLM, and the two
retrievers that genuinely require an LLM are demonstrated against a deterministic stub so you still see the
mechanism and the exact output shape. If you *do* have `ANTHROPIC_API_KEY` set, those cells run for real as well.

```bash
docker run -d --name extraction-sandbox-neo4j \
  -p 7690:7687 -p 7476:7474 \
  -e NEO4J_AUTH=neo4j/sandbox-kg \
  neo4j:5.26
```

Then browse the result at <http://localhost:7476>. Override the connection with `NEO4J_URI`, `NEO4J_USER`,
`NEO4J_PASSWORD` if yours lives elsewhere.

## 0. Connect

In [1]:
import os, sys, json, time, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", category=FutureWarning)

import neo4j, neo4j_graphrag, neo4j_viz, pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

import kgx
from kgx.data.documents import DOCUMENTS
from kgx.neo4j_io import (
    Neo4jConfig, connect, clear_database, load_graph, load_documents,
    counts, schema_summary, supports_dynamic_labels, has_apoc,
    drop_search_indexes, ENTITY_LABEL,
)

print(f"neo4j driver    {neo4j.__version__}")
print(f"neo4j-graphrag  {neo4j_graphrag.__version__}")
print(f"neo4j-viz       {neo4j_viz.__version__ if hasattr(neo4j_viz, '__version__') else '1.8.x'}")
print(f"kgx             {kgx.__version__}")

neo4j driver    6.2.0
neo4j-graphrag  1.18.0
neo4j-viz       1.8.x
kgx             0.1.0


In [2]:
# Override with NEO4J_URI / NEO4J_USER / NEO4J_PASSWORD if yours is elsewhere.
config = Neo4jConfig.from_env(
    uri=os.environ.get("NEO4J_URI", "bolt://localhost:7690"),
    user=os.environ.get("NEO4J_USER", "neo4j"),
    password=os.environ.get("NEO4J_PASSWORD", "sandbox-kg"),
)

if "driver" in globals():          # re-running this cell should not leak pools
    driver.close()
# notifications="OFF": exploring a relationship type the data does not contain is a
# legitimate move, and the server answers each one with a multi-line stderr warning.
driver = connect(config, notifications="OFF")

records, _, _ = driver.execute_query(
    "CALL dbms.components() YIELD name, versions, edition RETURN name, versions[0] AS version, edition")
print(f"connected to {config.uri}")
for r in records:
    print(f"  {r['name']} {r['version']} ({r['edition']})")
print(f"  dynamic labels $()  : {supports_dynamic_labels(driver)}")
print(f"  APOC installed      : {has_apoc(driver)}")

connected to bolt://localhost:7690
  Neo4j Kernel 5.26.19 (community)
  dynamic labels $()  : True
  APOC installed      : True


## 1. Loading a typed graph

The ontology decides the node labels at runtime — `company`, `business_segment`, `risk_factor` — and a plain
Cypher label cannot be parameterised. `MERGE (n:$label {...})` is a syntax error, not a query with a
parameter in it.

Historically that left two options: build the query string yourself (and own the injection risk), or reach
for APOC. Neo4j 5.24 added `SET n:$(label)` and 5.26 extended `$()` to `MERGE` and `MATCH`, so on a current
server the label is supplied *as data* and the whole load is one batched statement.

In [3]:
# The three shapes, on the actual server.
probes = [
    ("static label            ", "MERGE (n:ProbeStatic {id: 1}) RETURN labels(n) AS labels", {}),
    ("parameterised label     ", "MERGE (n:$label {id: 1}) RETURN labels(n) AS labels", {"label": "ProbeParam"}),
    ("dynamic label $()       ", "MERGE (n:$($label) {id: 1}) RETURN labels(n) AS labels", {"label": "ProbeDynamic"}),
    ("dynamic rel type $()    ", "MATCH (a:ProbeStatic), (b:ProbeDynamic) "
                                 "MERGE (a)-[r:$($rt)]->(b) RETURN type(r) AS labels", {"rt": "PROBE_REL"}),
]
for name, query, params in probes:
    try:
        recs, _, _ = driver.execute_query(query, **params)
        print(f"  {name} OK   -> {recs[0]['labels']}")
    except Exception as exc:
        print(f"  {name} FAIL -> {type(exc).__name__}: {str(exc).splitlines()[0][:90]}")

_ = driver.execute_query("MATCH (n) WHERE n:ProbeStatic OR n:ProbeParam OR n:ProbeDynamic DETACH DELETE n")

  static label             OK   -> ['ProbeStatic']
  parameterised label      FAIL -> CypherSyntaxError: {neo4j_code: Neo.ClientError.Statement.SyntaxError} {message: Invalid input 'label': expec
  dynamic label $()        OK   -> ['ProbeDynamic']
  dynamic rel type $()     OK   -> PROBE_REL


`$label` fails; `$($label)` works. That one extra pair of parentheses is the difference between a loader that
needs a plugin and one that does not.

`kgx.neo4j_io.load_graph` probes for the feature and falls back to grouping rows by label when it is absent —
one statement per label instead of one for everything. Both paths are batched with `UNWIND`; neither
concatenates user data into a query string.

> **Why not APOC?** `apoc.merge.node(labels, identProps, onCreateProps, onMatchProps)` does the same job, but
> the argument order is a trap: pass your properties as `onMatchProps` only — the obvious reading — and a
> *first* load writes nothing but the id. Every subsequent load looks correct, so the bug is invisible on any
> database you have already loaded once. This module hit that exact bug during development.

In [4]:
from kgx.graph import KnowledgeGraph

kg_path = ROOT / "output" / "business_news_kg.json"
if not kg_path.exists():
    raise FileNotFoundError(
        f"{kg_path} not found — run notebook 01 first, or rebuild with:\n"
        f"  uv run python -c \"import sys; sys.path.insert(0,'src'); ...\"\n"
        f"See notebook 01 §16 for the export cell."
    )

kg = KnowledgeGraph.from_json(kg_path, kgx.BUSINESS_NEWS)
print(kg.summary())

KnowledgeGraph: 109 entities, 68 edges
  node labels: {'geography': 23, 'financial_metric': 19, 'product': 10, 'business_segment': 9, 'company': 7, 'security': 7, 'business_event': 7, 'person': 7, 'sector': 6, 'risk_factor': 6, 'regulator': 5, 'commodity': 3}
  edge types:  {'operates_in': 16, 'reports_metric': 13, 'officer_of': 7, 'produces': 7, 'partners_with': 6, 'competes_with': 4, 'participant_in': 4, 'subsidiary_of': 4, 'faces_risk': 4, 'acquires': 2, 'impacts': 1}


### Clearing first

This wipes the target database. It is the right default for a demo — a load that half-overwrites an unrelated
graph is worse than one that starts clean — but check `config.uri` above before running it.

In [5]:
RESET = True   # set False to load alongside whatever is already there

if RESET:
    print(f"deleted {clear_database(driver)} nodes from {config.uri}")
    # Also drop search indexes. Neo4j rejects a second index on the same
    # (label, property), and CREATE ... IF NOT EXISTS treats an *equivalent*
    # index under a different name as already satisfied -- so it succeeds,
    # creates nothing, and the name you asked for never exists. §4 then fails
    # with "No index with name ... found" three cells later.
    print(f"dropped search indexes: {drop_search_indexes(driver) or 'none'}")

t = time.time()
load_stats = load_graph(driver, kg)
doc_stats = load_documents(driver, DOCUMENTS, kg)
print(f"\nloaded in {time.time()-t:.1f}s")
print(" ", load_stats)
print(" ", doc_stats)

deleted 119 nodes from bolt://localhost:7690
dropped search indexes: ['document_ft', 'document_vec', 'entity_ft', 'entity_vec']



loaded in 0.2s
  {'nodes': 109, 'relationships': 68, 'method': 'dynamic'}
  {'documents': 10, 'mention_links': 173}


Two things came in. The **graph** — canonical entities and their evidence-backed relationships — and the
**source documents** the graph was extracted from, linked by `MENTIONED_IN`.

That second half matters more than it looks. The extracted graph holds names and relations; it holds almost
no prose. Vector search needs text to search, and the interesting move in §5 is starting from a document hit
and walking into the structured graph. Without the documents there is nothing to walk from.

In [6]:
# Idempotency: MERGE on canon_id throughout, so re-running updates in place.
before = counts(driver)
load_graph(driver, kg)
load_documents(driver, DOCUMENTS, kg)
after = counts(driver)

print(f"before second load: {before['nodes']} nodes, {before['relationships']} relationships")
print(f"after  second load: {after['nodes']} nodes, {after['relationships']} relationships")
assert (before["nodes"], before["relationships"]) == (after["nodes"], after["relationships"])
print("\nidempotent — an extraction pipeline gets re-run constantly while the ontology is being tuned,")
print("and a loader that duplicates on every pass makes the graph useless as a working surface.")

before second load: 119 nodes, 241 relationships
after  second load: 119 nodes, 241 relationships

idempotent — an extraction pipeline gets re-run constantly while the ontology is being tuned,
and a loader that duplicates on every pass makes the graph useless as a working surface.


## 2. What actually landed

Before querying, look at the shape of what was written — read back from the data, not assumed from the
ontology.

In [7]:
c = counts(driver)
print(f"{c['nodes']} nodes, {c['relationships']} relationships\n")
display(pd.DataFrame([
    {"label": k, "count": v} for k, v in c["by_label"].items()
]).set_index("label").T)
display(pd.DataFrame([
    {"relationship": k, "count": v} for k, v in c["by_type"].items()
]).set_index("relationship").T)

119 nodes, 241 relationships



label,__Entity__,Geography,FinancialMetric,Product,Document,BusinessSegment,Company,Security,BusinessEvent,Person,Sector,RiskFactor,Regulator,Commodity
count,109,23,19,10,10,9,7,7,7,7,6,6,5,3


relationship,MENTIONED_IN,OPERATES_IN,REPORTS_METRIC,OFFICER_OF,PRODUCES,PARTNERS_WITH,COMPETES_WITH,PARTICIPANT_IN,SUBSIDIARY_OF,FACES_RISK,ACQUIRES,IMPACTS
count,173,16,13,7,7,6,4,4,4,4,2,1


In [8]:
# The real (head)-[rel]->(tail) patterns present in the data.
display(schema_summary(driver).head(15))

# db.schema.visualization() is the shortcut. It reads the token and count stores rather
# than the rows, so check it against the data before trusting it.
viz, _, _ = driver.execute_query("CALL db.schema.visualization()", routing_=neo4j.RoutingControl.READ)
labels = {n.element_id: n.labels for n in viz[0]["nodes"]}
reported = {(head, r.type, tail)
            for r in viz[0]["relationships"]
            for head in labels[r.start_node.element_id]
            for tail in labels[r.end_node.element_id]}
records, _, _ = driver.execute_query(
    "MATCH (h)-[r]->(t) UNWIND labels(h) AS head UNWIND labels(t) AS tail "
    "RETURN DISTINCT head, type(r) AS rel, tail", routing_=neo4j.RoutingControl.READ)
present = {(r["head"], r["rel"], r["tail"]) for r in records}
print(f"db.schema.visualization(): {len(reported)} label pairs, "
      f"{len(reported - present)} of them not in the data")

,head,relationship,tail,count
0,Geography,MENTIONED_IN,Document,31
1,Company,MENTIONED_IN,Document,27
2,FinancialMetric,MENTIONED_IN,Document,23
3,Person,MENTIONED_IN,Document,18
4,Company,OPERATES_IN,Geography,15
5,Product,MENTIONED_IN,Document,15
6,Company,REPORTS_METRIC,FinancialMetric,13
7,BusinessSegment,MENTIONED_IN,Document,12
8,Security,MENTIONED_IN,Document,10
9,Sector,MENTIONED_IN,Document,9


db.schema.visualization(): 65 label pairs, 0 of them not in the data


> `db.schema.visualization()` looks like the obvious way to get this, and it is worth the check above. It
> returns *virtual* nodes and relationships read from the token and count stores rather than from the rows,
> so what it reports and what the graph holds can drift — after a delete, most visibly, when the tokens
> outlive the data. On this graph they agree exactly, all 65 pairs. Counting the data is cheap and it cannot
> drift.

Note `MENTIONED_IN` accounts for most of the edges. That is the document linkage, not the knowledge — it
matters for retrieval and it will swamp any visualization, so §6 renders the entity subgraph separately.

In [9]:
# Every relationship carries the evidence that produced it.
records, _, _ = driver.execute_query("""
MATCH (c:Company)-[r]->(x:__Entity__)
WHERE c.name STARTS WITH 'Northwind'
RETURN type(r) AS relationship, x.name AS target,
       r.confidence AS confidence, r.support AS support, r.docs AS docs,
       r.evidence[0] AS evidence
ORDER BY r.support DESC, r.confidence DESC
LIMIT 8
""", routing_=neo4j.RoutingControl.READ)
display(pd.DataFrame([dict(r) for r in records]))

,relationship,target,confidence,support,docs,evidence
0,PARTICIPANT_IN,transaction,0.8628,8,[d01],ssets in 2026. The transaction has been unanimously appr...
1,REPORTS_METRIC,revenue,0.8697,7,"[d07, d09]",SEATTLE — Northwind Logistics cut its full-year revenue ...
2,PARTNERS_WITH,Cascade Freight Systems,0.5822,6,[d01],ted average price. Cascade Freight Systems operates 41 c...
3,ACQUIRES,Cascade Freight Systems,0.9434,5,"[d01, d04, d07]",SEATTLE — Northwind Logistics Inc. (NASDAQ: NWL) today a...
4,PARTNERS_WITH,Securities and Exchange Commission,0.9029,4,[d07],"01. On May 4, 2026, Northwind Logistics Inc. (the ""Compa..."
5,PARTICIPANT_IN,acquisition,0.7430,4,[d01],"xecutive Officer of Northwind Logistics. ""The proposed a..."
6,PARTNERS_WITH,Meridian Rail Group,0.7821,3,"[d02, d04]",and welcome to the Northwind Logistics fourth quarter an...
7,PARTICIPANT_IN,agreement,0.5639,3,[d01],"er the terms of the agreement, Cascade Freight sharehold..."


An edge you cannot trace to a sentence cannot be audited, and a graph built by a 194M-parameter model
contains wrong edges. `support` and `docs` are the triage columns: an edge asserted once in one document is a
hypothesis, an edge asserted across three documents is a fact.

## 3. Cypher: the questions that justify a graph

Direction convention first, because half of all graph-query bugs are direction bugs: in this graph
`(:Person)-[:OFFICER_OF]->(:Company)`, `(:Company)-[:ACQUIRES]->(:Company)` with the acquirer as head, and
`(:Company)-[:SUBSIDIARY_OF]->(:Company)` from subsidiary to parent.

In [10]:
def q(cypher, **params):
    """Run a read query and return a DataFrame."""
    records, _, _ = driver.execute_query(cypher, routing_=neo4j.RoutingControl.READ, **params)
    return pd.DataFrame([dict(r) for r in records])

display(q("""
MATCH (p:Person)-[:OFFICER_OF]->(c:Company)
OPTIONAL MATCH (c)-[:OPERATES_IN]->(g:Geography)
RETURN p.name AS person, c.name AS company, collect(DISTINCT g.name)[..4] AS operates_in
ORDER BY company, person
"""))

,person,company,operates_in
0,Ana Duarte,Halcyon Semiconductor Corporation,"[PHOENIX, Penang Malaysia, Dresden Germany, Phoenix Ariz..."
1,Elena Vasquez,Northwind Logistics Inc,"[Alberta, United States, Arizona, Canada]"
2,Marcus Webb,Northwind Logistics Inc,"[Alberta, United States, Arizona, Canada]"
3,Priya Raman,Northwind Logistics Inc,"[Alberta, United States, Arizona, Canada]"
4,Thomas Ingersoll,Northwind Logistics Inc,"[Alberta, United States, Arizona, Canada]"
5,Dana Okonkwo,Vantage Energy Partners,"[Arizona, Alberta, Texas]"
6,Marcus Webb,Vantage Energy Partners,"[Arizona, Alberta, Texas]"


### Scoped subqueries

`CALL (c) { ... }` — the *variable scope clause* — replaces the older `CALL { WITH c ... }`. On 5.26 the bare
`CALL {` form still runs but emits a deprecation notification on every call, which fills a notebook with
warnings.

In [11]:
display(q("""
MATCH (c:Company)
CALL (c) {
    MATCH (c)-[:OPERATES_IN]->(g:Geography) RETURN count(g) AS geographies
}
CALL (c) {
    MATCH (c)-[r]->(:__Entity__) WHERE type(r) <> 'MENTIONED_IN' RETURN count(r) AS out_edges
}
RETURN c.name AS company, c.n_docs AS docs, geographies, out_edges
ORDER BY out_edges DESC
"""))

,company,docs,geographies,out_edges
0,Northwind Logistics Inc,9,6,29
1,Halcyon Semiconductor Corporation,5,5,14
2,Vantage Energy Partners,2,3,7
3,Cascade Freight Systems,4,1,4
4,Torrent Microsystems,4,0,4
5,Meridian Rail Group,2,0,1
6,Dresden fabrication site,1,0,0


### Quantified path patterns

`((x)-[:SUBSIDIARY_OF]->(y)){1,3}` matches a chain of one to three hops. Before 5.9 this needed
variable-length syntax that could not constrain the intermediate nodes; now the repeated pattern is a
first-class thing and the whole chain comes back as one path.

Ownership chains are the canonical use: *who ultimately controls this company?*

In [12]:
display(q("""
MATCH path = (start:Company)((a)-[:SUBSIDIARY_OF]->(b)){1,3}(top:Company)
WHERE NOT (top)-[:SUBSIDIARY_OF]->()
RETURN start.name AS subsidiary,
       [n IN nodes(path) | n.name] AS ownership_chain,
       length(path) AS hops
ORDER BY hops DESC
"""))

,subsidiary,ownership_chain,hops
0,Torrent Microsystems,"[Torrent Microsystems, Northwind Logistics Inc, Halcyon ...",2
1,Cascade Freight Systems,"[Cascade Freight Systems, Northwind Logistics Inc, Halcy...",2
2,Meridian Rail Group,"[Meridian Rail Group, Northwind Logistics Inc, Halcyon S...",2
3,Northwind Logistics Inc,"[Northwind Logistics Inc, Halcyon Semiconductor Corporat...",1


Read that output sceptically — all four rows are wrong. `Northwind subsidiary_of Halcyon` is the edge
notebook 01 flagged as hallucinated: it is row 3 on its own and the second hop of the other three, so no
chain here survives it. Two of the first hops are independently bad. `Meridian subsidiary_of Northwind`
(support 1) is d02's 19.9% equity stake — *"a position it described as strategic rather than a prelude to
control"* — read as ownership. `Torrent subsidiary_of Northwind` (support 1) is a supplier read as a
subsidiary. Only `Cascade subsidiary_of Northwind` is real. **The Cypher is correct and the answer is wrong.**

This is what evidence properties are for. The same query, with the corroboration threshold applied to every
hop *and* to the test for an ultimate parent, drops all three fabricated edges — and with them every
multi-hop chain in the table. What survives is one corroborated hop: once Northwind's own parent edge is
disqualified, Northwind *is* the top of the chain. An edge asserted once in one document should not be
load-bearing in a multi-hop conclusion, and on this corpus nothing is left load-bearing at two hops.

In [13]:
display(q("""
MATCH path = (start:Company)((a)-[r:SUBSIDIARY_OF WHERE r.support >= 2 AND r.confidence >= 0.7]->(b)){1,3}(top:Company)
WHERE NOT EXISTS {
    (top)-[p:SUBSIDIARY_OF]->()
    WHERE p.support >= 2 AND p.confidence >= 0.7
}
RETURN start.name AS subsidiary,
       [n IN nodes(path) | n.name] AS ownership_chain,
       [rel IN relationships(path) | rel.support] AS support,
       length(path) AS hops
ORDER BY hops DESC
"""))
print("Filtering inside the quantified pattern prunes weak edges before the path is built,")
print("rather than assembling bad paths and discarding them afterwards. The ultimate-parent")
print("test gets the same threshold: a subsidiary edge too weak to traverse is also too weak")
print("to disqualify a node from being the top of the chain.")

,subsidiary,ownership_chain,support,hops
0,Cascade Freight Systems,"[Cascade Freight Systems, Northwind Logistics Inc]",[3],1


Filtering inside the quantified pattern prunes weak edges before the path is built,
rather than assembling bad paths and discarding them afterwards. The ultimate-parent
test gets the same threshold: a subsidiary edge too weak to traverse is also too weak
to disqualify a node from being the top of the chain.


### Multi-hop exposure — the query a vector store cannot answer

*Which companies are exposed to a firm under regulatory scrutiny, and by what route?* There is no passage to
retrieve. The answer only exists as a path.

In [14]:
# collect() here collapses parallel routes: four relationship types between the same
# pair would otherwise enumerate as four rows and crowd the second-order results out.
exposure = q("""
MATCH (at_risk:Company)-[:FACES_RISK]->(risk:RiskFactor)
MATCH path = (exposed:Company)-[:PARTNERS_WITH|COMPETES_WITH|SUBSIDIARY_OF|ACQUIRES*1..2]-(at_risk)
WHERE exposed <> at_risk
WITH exposed.name AS exposed_company, at_risk.name AS at_risk_company,
     risk.name AS risk_factor, length(path) AS hops,
     collect(DISTINCT [n IN nodes(path)[1..-1] | n.name][0]) AS through
RETURN exposed_company, at_risk_company, risk_factor, through, hops
ORDER BY hops DESC, exposed_company
""")
print(f"{len(exposure)} exposure routes  " +
      "  ".join(f"{h} hop(s): {n}" for h, n in exposure.hops.value_counts().sort_index().items()))
display(exposure.head(12))

31 exposure routes  1 hop(s): 15  2 hop(s): 16


,exposed_company,at_risk_company,risk_factor,through,hops
0,Cascade Freight Systems,Torrent Microsystems,price,[Northwind Logistics Inc],2
1,Cascade Freight Systems,Halcyon Semiconductor Corporation,fines,[Northwind Logistics Inc],2
2,Halcyon Semiconductor Corporation,Northwind Logistics Inc,supply disruptions,"[Torrent Microsystems, Vantage Energy Partners]",2
3,Halcyon Semiconductor Corporation,Torrent Microsystems,price,[Northwind Logistics Inc],2
4,Halcyon Semiconductor Corporation,Northwind Logistics Inc,spot rack prices,"[Torrent Microsystems, Vantage Energy Partners]",2
5,Meridian Rail Group,Torrent Microsystems,price,[Northwind Logistics Inc],2
6,Meridian Rail Group,Halcyon Semiconductor Corporation,fines,[Northwind Logistics Inc],2
7,Northwind Logistics Inc,Torrent Microsystems,price,[Halcyon Semiconductor Corporation],2
8,Northwind Logistics Inc,Halcyon Semiconductor Corporation,fines,"[Vantage Energy Partners, Torrent Microsystems]",2
9,Torrent Microsystems,Northwind Logistics Inc,supply disruptions,[Halcyon Semiconductor Corporation],2


Second-order exposure, ordered first because it is the interesting half: nobody wrote *"Cascade is exposed to
Halcyon's antitrust fines"* in any document, and no amount of semantic search over the text will surface it.
It exists only as a traversal — `through` names the intermediary that carries it, Northwind in that case.

Note also what is **absent**. The ontology declares `SUPPLIES`, `SUBJECT_TO` and `PARTY_TO`; the extraction
produced none of them on this corpus, so a query written against the ontology rather than against the data
returns nothing at all. Always check what actually landed — `schema_summary()` above — before writing the
interesting query.

In [15]:
# Shortest path between two entities, with the selector syntax (5.21+).
display(q("""
MATCH (a:Company {name: 'Halcyon Semiconductor Corporation'})
MATCH (b:Person {name: 'Priya Raman'})
MATCH path = ANY SHORTEST (a)-[r:!MENTIONED_IN]-{1,5}(b)
RETURN [n IN nodes(path) | coalesce(n.name, n.doc_id)] AS hops,
       [rel IN relationships(path) | type(rel)] AS via,
       length(path) AS length
"""))

,hops,via,length
0,"[Halcyon Semiconductor Corporation, Northwind Logistics ...","[PARTNERS_WITH, OFFICER_OF]",2


`-[r:!MENTIONED_IN]-` excludes the document-linkage edges. Leave them in and everything is two hops from
everything else through whichever document happened to mention both — a connectivity artefact of the corpus,
not a fact about the world. Worth remembering whenever a graph mixes structural edges with provenance edges.

In [16]:
# Aggregate risk view: which risks touch the most companies, and how confident are we?
display(q("""
MATCH (c:Company)-[r:FACES_RISK]->(risk:RiskFactor)
RETURN risk.name AS risk_factor,
       count(DISTINCT c) AS companies,
       collect(DISTINCT c.name) AS which,
       round(avg(r.confidence), 3) AS avg_confidence,
       sum(r.support) AS total_support
ORDER BY companies DESC, total_support DESC
"""))

,risk_factor,companies,which,avg_confidence,total_support
0,fines,1,[Halcyon Semiconductor Corporation],0.962,5
1,spot rack prices,1,[Northwind Logistics Inc],0.735,2
2,supply disruptions,1,[Northwind Logistics Inc],0.919,1
3,price,1,[Torrent Microsystems],0.386,1


## 4. Indexes and embeddings

The Cypher above is all *structural* — it needs exact names and exact relationship types. Retrieval is the
other half: finding the right starting point from a question phrased however the user phrased it.

That needs two index families, and one modelling decision first.

### One shared label

Cypher indexes are per label. This graph has twelve ontology labels, so "find the entity most similar to this
question" would need twelve vector indexes and a twelve-way union at query time.

The fix is a shared secondary label. Every canonical entity carries `:__Entity__` alongside its ontology
label, so one index covers all of them and the specific label is still there for the structural queries.
`kgx`'s loader adds it during the load.

> The tempting alternative — `create_fulltext_index(..., label="Company|Person")` — **silently does the wrong
> thing.** It succeeds and creates an index on one literal label named `Company|Person`, which nothing has. It
> matches nothing, forever, without an error.

In [17]:
display(q("""
MATCH (n:__Entity__)
RETURN [l IN labels(n) WHERE l <> '__Entity__'][0] AS ontology_label, count(*) AS n
ORDER BY n DESC
"""))
print(f"every entity carries :__Entity__ plus its ontology label")

,ontology_label,n
0,Geography,23
1,FinancialMetric,19
2,Product,10
3,BusinessSegment,9
4,Company,7
5,Security,7
6,BusinessEvent,7
7,Person,7
8,Sector,6
9,RiskFactor,6


every entity carries :__Entity__ plus its ontology label


In [18]:
from neo4j_graphrag.embeddings import SentenceTransformerEmbeddings

embedder = SentenceTransformerEmbeddings("all-MiniLM-L6-v2")   # ~90 MB, local, no API key
DIM = len(embedder.embed_query("dimension probe"))
print(f"embedding dimension: {DIM}")

# Guard it. The stock examples in most docs use 1536 (OpenAI), and a mismatch here
# does not fail loudly -- the index is created at the wrong width and search returns
# nothing useful.
assert DIM == 384, f"expected 384 for all-MiniLM-L6-v2, got {DIM}"

embedding dimension: 384


In [19]:
from neo4j_graphrag.indexes import create_vector_index, create_fulltext_index, upsert_vectors
from neo4j_graphrag.types import EntityType

INDEXES = {
    "document_vec": dict(kind="vector",   label="Document",   prop="embedding"),
    "document_ft":  dict(kind="fulltext", label="Document",   props=["title", "text"]),
    "entity_vec":   dict(kind="vector",   label=ENTITY_LABEL, prop="embedding"),
    "entity_ft":    dict(kind="fulltext", label=ENTITY_LABEL, props=["name", "aliases"]),
}

for name, spec in INDEXES.items():
    if spec["kind"] == "vector":
        create_vector_index(driver, name, label=spec["label"], embedding_property=spec["prop"],
                            dimensions=DIM, similarity_fn="cosine")
    else:
        create_fulltext_index(driver, name, label=spec["label"], node_properties=spec["props"])
    print(f"  {name:14} {spec['kind']:9} on :{spec['label']}")

  document_vec   vector    on :Document
  document_ft    fulltext  on :Document
  entity_vec     vector    on :__Entity__
  entity_ft      fulltext  on :__Entity__


Four indexes, two per target. **Documents** carry the prose; **entities** carry names and aliases. They answer
different questions and §5 uses both.

Each target gets a vector index *and* a full-text index because they fail in opposite directions. Vectors
handle paraphrase and miss exact tokens — a ticker symbol embeds into nothing in particular. Full-text nails
`NWL` and answers *"the freight company being investigated"* with whatever node happens to be named
`freight`. Hybrid retrieval exists because neither is sufficient.

In [20]:
# Embed the documents: title + body.
records, _, _ = driver.execute_query(
    "MATCH (d:Document) RETURN elementId(d) AS eid, d.title + '\n' + d.text AS text",
    routing_=neo4j.RoutingControl.READ)
upsert_vectors(driver, ids=[r["eid"] for r in records], embedding_property="embedding",
               embeddings=[embedder.embed_query(r["text"]) for r in records],
               entity_type=EntityType.NODE)
print(f"embedded {len(records)} documents")

# Embed the entities: type, canonical name, aliases. The aliases are the entity-resolution
# work from notebook 01 paying off a second time -- they widen what the entity matches.
records, _, _ = driver.execute_query("""
MATCH (n:__Entity__)
RETURN elementId(n) AS eid,
       n.type + ': ' + n.name +
       CASE WHEN size(n.aliases) > 0
            THEN ' (also known as ' + reduce(s = head(n.aliases), a IN tail(n.aliases) | s + ', ' + a) + ')'
            ELSE '' END AS text
""", routing_=neo4j.RoutingControl.READ)
upsert_vectors(driver, ids=[r["eid"] for r in records], embedding_property="embedding",
               embeddings=[embedder.embed_query(r["text"]) for r in records],
               entity_type=EntityType.NODE)
print(f"embedded {len(records)} entities")
print("\nsample entity text embedded:")
print(" ", records[0]["text"])

embedded 10 documents


embedded 109 entities

sample entity text embedded:
  geography: Seattle (also known as SEATTLE)


### Indexes come online asynchronously

`create_vector_index` returns before the index is queryable. Construct a retriever immediately afterwards and
it either raises *"No index with name … found"* or, worse, succeeds against a half-populated index and returns
plausible but incomplete results.

At this scale population takes milliseconds. Poll anyway — the point is that the wait is deterministic rather
than a race you usually win.

In [21]:
def await_indexes(driver, names, timeout=60.0, database=None):
    """Block until every named index reports ONLINE."""
    deadline = time.time() + timeout
    while True:
        records, _, _ = driver.execute_query(
            "SHOW INDEXES YIELD name, state, populationPercent "
            "WHERE name IN $names RETURN name, state, populationPercent",
            names=list(names), routing_=neo4j.RoutingControl.READ)
        states = {r["name"]: (r["state"], r["populationPercent"]) for r in records}
        if len(states) == len(names) and all(s == "ONLINE" for s, _ in states.values()):
            return states
        if time.time() > deadline:
            raise TimeoutError(f"indexes not ONLINE within {timeout}s: {states}")
        time.sleep(0.2)

t = time.time()
states = await_indexes(driver, INDEXES)
print(f"all four ONLINE after {time.time()-t:.2f}s")
for name, (state, pct) in states.items():
    print(f"  {name:14} {state}  {pct:.0f}%")

all four ONLINE after 0.01s
  document_ft    ONLINE  100%
  document_vec   ONLINE  100%
  entity_ft      ONLINE  100%
  entity_vec     ONLINE  100%


## 5. The retrievers

`neo4j-graphrag` ships six retrievers. They are not six flavours of the same thing — they differ in what they
search, what they return, and what they require.

| retriever | searches | needs | returns |
|---|---|---|---|
| `VectorRetriever` | one vector index | embedder | the matched nodes' properties |
| `VectorCypherRetriever` | one vector index, then **traverses** | embedder | whatever your Cypher returns |
| `HybridRetriever` | vector **+** full-text | embedder | the matched nodes' properties |
| `HybridCypherRetriever` | vector **+** full-text, then traverses | embedder | whatever your Cypher returns |
| `Text2CypherRetriever` | nothing — **writes Cypher** | **LLM** | query results |
| `ToolsRetriever` | routes to other retrievers | **tool-calling LLM** | the chosen retriever's results |

The first four need no LLM at all. Only the last two do, and those are stubbed below so the notebook runs
without credentials.

In [22]:
from neo4j_graphrag.retrievers import (
    VectorRetriever, VectorCypherRetriever, HybridRetriever,
    HybridCypherRetriever, Text2CypherRetriever, ToolsRetriever,
)

def show(result, label, width=150):
    """Print a RetrieverResult compactly."""
    print(f"── {label}  ({len(result.items)} items)")
    for item in result.items:
        score = (item.metadata or {}).get("score")
        prefix = f"  [{score:.3f}] " if isinstance(score, float) else "  "
        print(prefix + str(item.content).replace("\n", " ")[:width])
    print()

### 5.1 `VectorRetriever` — semantic search over documents

In [23]:
doc_vector = VectorRetriever(driver, index_name="document_vec", embedder=embedder,
                             return_properties=["doc_id", "title"])

show(doc_vector.search(query_text="Which company is being investigated by regulators over its accounting?",
                       top_k=3),
     "accounting investigation")
show(doc_vector.search(query_text="chip export restrictions and semiconductor pricing", top_k=3),
     "semiconductor pricing")

── accounting investigation  (3 items)
  [0.695] {'title': 'Northwind Logistics Inc. Form 8-K — Item 8.01 Other Events', 'doc_id': 'd07'}
  [0.688] {'title': 'Halcyon Semiconductor Corporation Second Quarter 2026 Earnings Call', 'doc_id': 'd10'}
  [0.674] {'title': 'EU opens antitrust inquiry into Halcyon as price war with Torrent intensifies', 'doc_id': 'd06'}

── semiconductor pricing  (3 items)
  [0.710] {'title': 'EU opens antitrust inquiry into Halcyon as price war with Torrent intensifies', 'doc_id': 'd06'}
  [0.655] {'title': 'Halcyon Semiconductor Corporation Second Quarter 2026 Earnings Call', 'doc_id': 'd10'}
  [0.651] {'title': 'Halcyon Semiconductor Names Ana Duarte Chief Executive Officer', 'doc_id': 'd05'}



Neo4j normalises cosine similarity to `(1 + cos) / 2`, so the scale is `[0, 1]` with **0.5 meaning
orthogonal** — not "half similar". That also compresses the range, and the compressed range carries almost no
relevance signal: §5.4 returns three *wrong* documents at 0.628, 0.604 and 0.588, inside the same band as the
hits above. Read the ordering, not the number, and do not carry a threshold over from raw
sentence-transformers cosine values.

### 5.2 `VectorRetriever` over entities — a different question entirely

In [24]:
entity_vector = VectorRetriever(driver, index_name="entity_vec", embedder=embedder,
                                return_properties=["name", "type", "n_docs"])

show(entity_vector.search(query_text="a rail freight partner", top_k=4), "a rail freight partner")
show(entity_vector.search(query_text="something that could hurt margins", top_k=4), "something that could hurt margins")

── a rail freight partner  (4 items)
  [0.805] {'name': 'freight brokerage', 'n_docs': 1, 'type': 'sector'}
  [0.801] {'name': 'Meridian Rail Group', 'n_docs': 2, 'type': 'company'}
  [0.800] {'name': 'freight', 'n_docs': 4, 'type': 'sector'}
  [0.786] {'name': 'Cascade Freight Systems', 'n_docs': 4, 'type': 'company'}

── something that could hurt margins  (4 items)
  [0.740] {'name': 'margins', 'n_docs': 1, 'type': 'financial_metric'}
  [0.703] {'name': 'margin', 'n_docs': 1, 'type': 'financial_metric'}
  [0.688] {'name': 'Adjusted operating margin', 'n_docs': 1, 'type': 'financial_metric'}
  [0.681] {'name': 'operating margin', 'n_docs': 1, 'type': 'financial_metric'}



Same retriever class, same index type, completely different use. Document search answers *"where should I
read?"*; entity search answers *"which node in the graph does this phrase mean?"* — the entry point for any
structural query. Note how it answers, though: *"a rail freight partner"* puts the `freight brokerage` sector
node above `Meridian Rail Group`, with all four hits inside 0.02 of each other. What is embedded per entity
is a name and its aliases, not a description, so a descriptive phrase lands on whichever node is *named* like
the description. §5.5 pushes on that and finds the edge of it.

### 5.3 `VectorCypherRetriever` — the one that makes it GraphRAG

This is the retriever worth understanding. Vector search finds a starting node; then **your Cypher runs from
it**, and whatever that returns becomes the context. The retrieved context is no longer limited to text that
exists in one place.

In [25]:
# `facts` is collected per entity, so the second WITH holds a list of lists. reduce()
# flattens it without APOC -- the setup at the top of this notebook installs no plugins,
# and the retrieval query should not need one either.
DOC_TO_GRAPH = """
MATCH (node)<-[:MENTIONED_IN]-(e:__Entity__)
OPTIONAL MATCH (e)-[r]->(o:__Entity__) WHERE type(r) <> 'MENTIONED_IN'
WITH node, e, collect(DISTINCT e.name + ' -[' + type(r) + ']-> ' + o.name) AS facts
WITH node, collect(DISTINCT e.name) AS entities, collect(facts) AS fact_lists
RETURN node.doc_id AS doc_id, node.title AS title,
       entities[..8] AS entities,
       reduce(acc = [], l IN fact_lists | acc + l)[..8] AS facts
"""

def format_doc_context(record):
    from neo4j_graphrag.types import RetrieverResultItem
    return RetrieverResultItem(
        content=(f"[{record['doc_id']}] {record['title']}\n"
                 f"    entities: {', '.join(record['entities'])}\n"
                 f"    facts:    " + "; ".join(record["facts"][:5])),
        metadata={"doc_id": record["doc_id"]},
    )

doc_graph = VectorCypherRetriever(driver, index_name="document_vec", embedder=embedder,
                                  retrieval_query=DOC_TO_GRAPH,
                                  result_formatter=format_doc_context)

result = doc_graph.search(query_text="Which company is being investigated by regulators?", top_k=2)
for item in result.items:
    print(item.content, "\n")

# Nothing in that query orders the facts or filters on evidence. How big is the pool
# of one-document edges it draws from, and what does one of them look like?
display(q("""
MATCH (a:__Entity__)-[r]->(b:__Entity__) WHERE type(r) <> 'MENTIONED_IN'
RETURN count(*) AS entity_edges,
       sum(CASE WHEN r.support = 1 THEN 1 ELSE 0 END) AS support_1,
       sum(CASE WHEN r.support = 1 AND r.confidence >= 0.7 THEN 1 ELSE 0 END) AS support_1_confident
"""))
display(q("""
MATCH (p:Person {name: 'Marcus Webb'})-[r:OFFICER_OF]->(c:Company)
RETURN c.name AS employer, r.support AS support, round(r.confidence, 3) AS confidence,
       r.docs AS docs, left(r.evidence[0], 60) AS evidence
ORDER BY r.support DESC
"""))

[d06] EU opens antitrust inquiry into Halcyon as price war with Torrent intensifies
    entities: fines, controller boards, radar front-ends, Seattle, Northwind Logistics Inc, Ana Duarte, freight, Shares
    facts:    Northwind Logistics Inc -[OPERATES_IN]-> Alberta; Northwind Logistics Inc -[PARTICIPANT_IN]-> agreement; Northwind Logistics Inc -[OPERATES_IN]-> United States; Northwind Logistics Inc -[REPORTS_METRIC]-> operating margin; Northwind Logistics Inc -[REPORTS_METRIC]-> margin 

[d07] Northwind Logistics Inc. Form 8-K — Item 8.01 Other Events
    entities: Securities and Exchange Commission, Surface Transportation Board, Priya Raman, Marcus Webb, Northwind Logistics Inc, United States District Court, revenue, Cascade Freight transaction
    facts:    Priya Raman -[OFFICER_OF]-> Northwind Logistics Inc; Marcus Webb -[OFFICER_OF]-> Vantage Energy Partners; Marcus Webb -[OFFICER_OF]-> Northwind Logistics Inc; Northwind Logistics Inc -[OPERATES_IN]-> Alberta; Northwind Logistics 

,entity_edges,support_1,support_1_confident
0,68,24,7


,employer,support,confidence,docs,evidence
0,Northwind Logistics Inc,4,0.973,"[d01, d04, d07, d09]",SEATTLE — Northwind Logistics cut its full-year revenue ...
1,Vantage Energy Partners,1,0.962,[d08],"nder the agreement, Vantage Energy Partners will supply ..."


Compare that to §5.1, which returned a title and an id. Here a single vector hit brings back the document
*plus* the entities extracted from it *plus* their outgoing relationships — structured facts that appear
nowhere as a contiguous span of text.

**Always pass a `result_formatter`** to the `*Cypher` retrievers. The default is `content=str(record)`, which
hands the LLM `"<Record doc_id='d07' title=… entities=[…]>"` — valid, ugly, and a waste of context.

Two things decide what the model actually sees, and neither is cosmetic. The lists are truncated twice —
`[..8]` in the Cypher, `[:5]` in the formatter — and *nothing orders them*, so storage order picks which five
facts reach the prompt. Reload the database and a different five come back — which is not a thought
experiment, it happened while this section was being written.

Nor does anything filter on evidence, and the pool it samples from is not clean: 24 of the 68 entity edges
rest on a single document, 7 of those at confidence above 0.7, so a confidence threshold will not save you
either. The d07 slice above is carrying one right now — `Marcus Webb -[OFFICER_OF]-> Vantage Energy
Partners` — and the second table says why that matters. Webb is Northwind's CFO across four documents; he is
an officer of Vantage on the strength of one sentence about a diesel supply agreement, extracted at 0.962
confidence. A retrieval query feeding an LLM carries the same obligation §3's path query did, and
`AND r.support >= 2` on the `OPTIONAL MATCH` is the whole change. It is left off here so the cost stays
visible.

### 5.4 `HybridRetriever` — where vectors alone fail

In [26]:
doc_hybrid = HybridRetriever(driver, vector_index_name="document_vec",
                             fulltext_index_name="document_ft", embedder=embedder,
                             return_properties=["doc_id", "title"])

QUERY = "STB review"
show(doc_vector.search(query_text=QUERY, top_k=3), f"vector only  — {QUERY!r}")
show(doc_hybrid.search(query_text=QUERY, top_k=3), f"hybrid       — {QUERY!r}")

# Where does the right document actually sit in the vector ranking? The whole corpus:
show(doc_vector.search(query_text=QUERY, top_k=10), f"vector only, all 10 — {QUERY!r}", width=64)

── vector only  — 'STB review'  (3 items)
  [0.628] {'title': 'EU opens antitrust inquiry into Halcyon as price war with Torrent intensifies', 'doc_id': 'd06'}
  [0.604] {'title': 'Halcyon Semiconductor Corporation Second Quarter 2026 Earnings Call', 'doc_id': 'd10'}
  [0.588] {'title': 'Northwind Logistics Inc. Form 8-K — Item 8.01 Other Events', 'doc_id': 'd07'}



── hybrid       — 'STB review'  (3 items)
  [1.000] {'title': 'US regulator opens review of Northwind-Cascade freight deal as Meridian pact expands', 'doc_id': 'd02'}
  [1.000] {'title': 'EU opens antitrust inquiry into Halcyon as price war with Torrent intensifies', 'doc_id': 'd06'}
  [0.961] {'title': 'Halcyon Semiconductor Corporation Second Quarter 2026 Earnings Call', 'doc_id': 'd10'}

── vector only, all 10 — 'STB review'  (10 items)
  [0.628] {'title': 'EU opens antitrust inquiry into Halcyon as price war 
  [0.604] {'title': 'Halcyon Semiconductor Corporation Second Quarter 2026
  [0.588] {'title': 'Northwind Logistics Inc. Form 8-K — Item 8.01 Other E
  [0.584] {'title': 'US regulator opens review of Northwind-Cascade freigh
  [0.576] {'title': 'Northwind Logistics Inc. Form 8-K — Item 5.02 Directo
  [0.576] {'title': 'Halcyon Semiconductor Names Ana Duarte Chief Executiv
  [0.571] {'title': 'Northwind cuts 2026 guidance as diesel costs bite; sh
  [0.567] {'title': 'Northwind 

`STB` is the Surface Transportation Board. To MiniLM it is three letters with no meaning, so at `top_k=3`
pure vector search misses **d02** — the document that is actually about the STB opening a review. Look at the
ten-document ranking, though: d02 is not absent from it, only fourth, at 0.584 against the 0.588 that made
the cut. The full-text half scores d02 far above anything else and pulls it to the top.

Three honest caveats, because this effect is smaller than the usual telling of it:

- On most queries here the two rankings **agree**. Hybrid is insurance against a specific failure mode, not a
  general uplift, and on a corpus this small the same document usually wins either way.
- Vector search is not helpless with rare tokens. `'NWL subpoena'` returns the right document from the vector
  index alone, because the rest of the query carries enough signal. The acronym only decides the outcome when
  it is *all* the signal there is.
- The margin is noise. Four thousandths separate d02 from the document above it, in a ten-document ranking
  whose entire spread is 0.07. A wider `top_k` would have found it; what hybrid buys is making it **first**,
  by a real margin rather than a rounding error.

Hybrid combines the two rankings, and *how* it combines them is a parameter worth knowing about.

In [27]:
from neo4j_graphrag.types import HybridSearchRanker

for label, kwargs in [
    ("naive  (max of the two normalised scores)", dict(ranker=HybridSearchRanker.NAIVE)),
    ("linear (alpha=0.9, vector-weighted)",       dict(ranker=HybridSearchRanker.LINEAR, alpha=0.9)),
    ("linear (alpha=0.1, fulltext-weighted)",     dict(ranker=HybridSearchRanker.LINEAR, alpha=0.1)),
]:
    res = doc_hybrid.search(query_text=QUERY, top_k=3, **kwargs)
    scores = [f"{(i.metadata or {}).get('score', 0):.3f}" for i in res.items]
    titles = [str(i.content)[:52] for i in res.items]
    print(f"  {label}")
    for s, t in zip(scores, titles):
        print(f"      {s}  {t}")
    print()

  naive  (max of the two normalised scores)
      1.000  {'title': 'US regulator opens review of Northwind-Ca
      1.000  {'title': 'EU opens antitrust inquiry into Halcyon a
      0.961  {'title': 'Halcyon Semiconductor Corporation Second 



  linear (alpha=0.9, vector-weighted)
      0.900  {'title': 'EU opens antitrust inquiry into Halcyon a
      0.886  {'title': 'Northwind Logistics Inc. Form 8-K — Item 
      0.865  {'title': 'Halcyon Semiconductor Corporation Second 

  linear (alpha=0.1, fulltext-weighted)
      0.900  {'title': 'US regulator opens review of Northwind-Ca
      0.490  {'title': 'Northwind Logistics Inc. Form 8-K — Item 
      0.293  {'title': 'Northwind cuts 2026 guidance as diesel co



`naive` takes the max of the two normalised scores, which saturates at 1.0 and produces ties. `linear` with
an `alpha` gives a real spread and lets you dial the balance — `alpha` is **required** with `linear`, and
passing it with `naive` warns and ignores it.

Note also: `filters` is a vector-family parameter. `HybridRetriever` and `HybridCypherRetriever` do not accept
it at all.

### 5.5 `HybridCypherRetriever` on entities — where the entity resolution pays off

In [28]:
ENTITY_NEIGHBOURHOOD = """
MATCH (node)
OPTIONAL MATCH (node)-[r]->(o:__Entity__) WHERE type(r) <> 'MENTIONED_IN'
RETURN node.name AS name, node.type AS type, node.aliases AS aliases,
       collect(type(r) + ' -> ' + o.name)[..6] AS relations
"""

def format_entity(record):
    from neo4j_graphrag.types import RetrieverResultItem
    aliases = f"  (aka {', '.join(record['aliases'])})" if record["aliases"] else ""
    return RetrieverResultItem(
        content=f"({record['type']}) {record['name']}{aliases}\n      " + "\n      ".join(record["relations"]),
        metadata={"name": record["name"]},
    )

entity_hybrid = HybridCypherRetriever(driver, vector_index_name="entity_vec",
                                      fulltext_index_name="entity_ft", embedder=embedder,
                                      retrieval_query=ENTITY_NEIGHBOURHOOD,
                                      result_formatter=format_entity)

# Looking an entity up by name is a LEXICAL task, so weight the full-text half:
# alpha is the vector weight, and 0.2 means "mostly trust the literal match".
LOOKUP = dict(ranker="linear", alpha=0.2)
DESCRIPTIVE = "the freight company that made an acquisition"

for query in ["NWL", "Mr. Webb", DESCRIPTIVE]:
    print(f"── {query!r}")
    for item in entity_hybrid.search(query_text=query, top_k=2, **LOOKUP).items:
        print("  " + item.content.replace("\n", "\n  "))
    print()

# The third one fails. Blame the lexical weighting? Re-run it vector-weighted.
print(f"── {DESCRIPTIVE!r}  at alpha=0.9")
for item in entity_hybrid.search(query_text=DESCRIPTIVE, top_k=2, ranker="linear", alpha=0.9).items:
    print("  " + item.content.replace("\n", "\n  "))

── 'NWL'


  (company) Northwind Logistics Inc  (aka NWL, Northwind, Northwind Logistics, Northwind Logistics Inc.)
        OPERATES_IN -> Alberta
        PARTICIPANT_IN -> agreement
        OPERATES_IN -> United States
        REPORTS_METRIC -> operating margin
        REPORTS_METRIC -> margin
        OPERATES_IN -> Arizona
  (geography) Dresden
        

── 'Mr. Webb'
  (person) Marcus Webb
        OFFICER_OF -> Vantage Energy Partners
        OFFICER_OF -> Northwind Logistics Inc
  (person) Robert Iyer
        

── 'the freight company that made an acquisition'


  (sector) freight  (aka Freight)
        
  (business_event) acquisition
        

── 'the freight company that made an acquisition'  at alpha=0.9
  (sector) freight  (aka Freight)
        
  (company) Cascade Freight Systems  (aka Cascade)
        OPERATES_IN -> Pacific Northwest
        REPORTS_METRIC -> revenue
        SUBSIDIARY_OF -> Northwind Logistics Inc
        COMPETES_WITH -> Northwind Logistics Inc


In [29]:
# Why alpha=0.2 rather than the default? Look at the halves separately, with scores.
# (A plain HybridRetriever, so `show` can print the score the formatter above drops.)
show(entity_vector.search(query_text="NWL", top_k=4), "NWL — the vector half alone", width=60)

entity_plain = HybridRetriever(driver, vector_index_name="entity_vec",
                               fulltext_index_name="entity_ft", embedder=embedder,
                               return_properties=["name"])
for label, kwargs in [("naive, the default", {}),
                      ("linear, alpha=0.2 ", dict(ranker="linear", alpha=0.2)),
                      ("linear, alpha=0.9 ", dict(ranker="linear", alpha=0.9))]:
    show(entity_plain.search(query_text="NWL", top_k=4, **kwargs), f"NWL — {label}", width=60)

── NWL — the vector half alone  (4 items)
  [0.615] {'name': 'Dresden', 'n_docs': 1, 'type': 'geography'}
  [0.615] {'name': 'HLCN', 'n_docs': 2, 'type': 'security'}
  [0.615] {'name': 'Dresden facility', 'n_docs': 1, 'type': 'business_
  [0.602] {'name': 'Dresden fabrication site', 'n_docs': 1, 'type': 'c



── NWL — naive, the default  (4 items)
  [1.000] {'name': 'Northwind Logistics Inc'}
  [1.000] {'name': 'Dresden'}
  [1.000] {'name': 'HLCN'}
  [0.999] {'name': 'Dresden facility'}



── NWL — linear, alpha=0.2   (4 items)
  [0.800] {'name': 'Northwind Logistics Inc'}
  [0.200] {'name': 'Dresden'}
  [0.200] {'name': 'HLCN'}
  [0.200] {'name': 'Dresden facility'}

── NWL — linear, alpha=0.9   (4 items)
  [0.900] {'name': 'Dresden'}
  [0.900] {'name': 'HLCN'}
  [0.899] {'name': 'Dresden facility'}
  [0.881] {'name': 'Dresden fabrication site'}



`NWL` resolves to **Northwind Logistics Inc** and `Mr. Webb` to **Marcus Webb** — neither query string is the
node's `name`, but they get there by different routes.

The `alpha=0.2` matters, and the four rankings above show how much. Three characters give the embedding
nothing to work with: the vector half alone answers `NWL` with *Dresden*, *HLCN* and *Dresden facility*, all
three at 0.615 and none of them Northwind. The naive ranker takes the max of the two normalised halves, so it
hands that noise a 1.000 and ties it with the perfect lexical match — which of the three tied hits leads is
arbitrary. `linear` at `alpha=0.2` separates them, 0.800 against 0.200. Push alpha the other way and it is
worse than arbitrary: at 0.9 the noise wins outright and Northwind is not in the top four at all. Name lookup
is a lexical task; weight it that way.

`NWL` matches because the `aliases` list is in the full-text index, and that list is exactly what notebook
01's entity resolution produced when it collapsed `NWL` / `Northwind` / `Northwind Logistics` /
`Northwind Logistics Inc.` into one node — the `(aka …)` line above Northwind is that list. `Mr. Webb` is the
cheaper case, and the missing `(aka …)` on Marcus Webb says so: he has no aliases at all. The Lucene analyzer
tokenises `Mr. Webb` to `[mr, webb]`, `webb` is already a token of the indexed `name`, and full-text would
have found him with no resolution underneath.

The third query is the one that does not work, and it is worth sitting with. *"the freight company that made
an acquisition"* returns the `freight` sector node and the `acquisition` event node — the query's own literal
tokens — where §5.2 promised `Northwind Logistics Inc`. Vector-weighting it does not help: at `alpha=0.9` the
answer is `freight` and *Cascade Freight Systems*, the company that was acquired rather than the one that did
the acquiring. The alpha is not the problem. What is indexed for each entity is
`type: name (also known as …)` from §4, and nothing in Northwind's embedded text says it is a freight company
or that it bought anything. A description matches a *document*; only a name matches an entity. Route the
query to the right index before it gets there.

An extraction pipeline without entity resolution cannot do the `NWL` case at all. There would be no alias
list to index, four separate Northwind nodes to choose between, and `NWL` would be an orphan.

The counter-example is worth knowing too: `HLCN` does **not** resolve to Halcyon Semiconductor here. The
extractor typed it as a `security` rather than as an alias of the company, so it became its own node. The
retrieval is only ever as good as the resolution underneath it, and this is what that looks like when it goes
wrong — not an error, just a slightly wrong answer with no indication anything happened.

### 5.6 The two retrievers that need an LLM

`Text2CypherRetriever` and `ToolsRetriever` cannot work without one — the first *writes Cypher*, the second
*chooses a retriever*. Both are language tasks, not search tasks.

So the notebook runs them against a deterministic stub. That is not a cop-out: the stub shows the mechanism,
the exact prompt the retriever assembles, and the exact result shape — which is most of what you need to know
— and if `ANTHROPIC_API_KEY` is set, every one of these cells also runs for real.

In [30]:
API_KEY = os.environ.get("ANTHROPIC_API_KEY")
live_llm = None

if API_KEY:
    try:
        from neo4j_graphrag.llm import AnthropicLLM
        live_llm = AnthropicLLM(model_name="claude-sonnet-4-5", model_params={"max_tokens": 1024})
        print(f"live LLM enabled: {type(live_llm).__name__}")
    except ImportError as exc:
        print(f"ANTHROPIC_API_KEY is set but the client is missing ({exc}).")
        print('  uv add "neo4j-graphrag[anthropic]"')
else:
    print("No ANTHROPIC_API_KEY — LLM cells run against the offline stub below.")
    print("  For real answers: export ANTHROPIC_API_KEY=... and uv add \"neo4j-graphrag[anthropic]\"")

No ANTHROPIC_API_KEY — LLM cells run against the offline stub below.
  For real answers: export ANTHROPIC_API_KEY=... and uv add "neo4j-graphrag[anthropic]"


In [31]:
from typing import Any, Optional, Sequence, Union
from neo4j_graphrag.llm import LLMBase, LLMResponse
from neo4j_graphrag.llm.types import ToolCall, ToolCallResponse


class EchoLLM(LLMBase):
    """A deterministic, offline stand-in for an LLM.

    Subclass `LLMBase`, not `LLMInterface`. `GraphRAG` dispatches on isinstance,
    and a duck-typed object with an `.invoke` method passes the constructor's
    validation and then raises "Type ... of LLM is not supported" at search time —
    a failure that arrives several cells after the mistake.

    `canned` makes it return a fixed string, which is how a scripted Cypher query
    or a scripted tool choice gets demonstrated without a model.
    """

    def __init__(self, model_name: str = "echo-stub", canned: Optional[str] = None,
                 tool_plan: Sequence[tuple[str, dict]] = (), **kwargs: Any) -> None:
        super().__init__(model_name=model_name, **kwargs)
        self.canned = canned
        self.tool_plan = list(tool_plan)
        self.prompts: list[str] = []          # so we can show what was actually sent

    @staticmethod
    def _flatten(value) -> str:
        if isinstance(value, str):
            return value                                   # v1 path (Text2CypherRetriever)
        return "\n".join(f"{m['role']}: {m['content']}" for m in value)   # v2 path (GraphRAG)

    def invoke(self, input, message_history=None, system_instruction=None, **kwargs) -> LLMResponse:
        prompt = self._flatten(input)
        self.prompts.append(prompt)
        content = self.canned if self.canned is not None else (
            f"[{self.model_name}] received {len(prompt)} characters of prompt; "
            f"a real model would answer here."
        )
        return LLMResponse(content=content)

    async def ainvoke(self, input, message_history=None, system_instruction=None, **kwargs) -> LLMResponse:
        return self.invoke(input, message_history, system_instruction, **kwargs)

    def invoke_with_tools(self, input, tools, message_history=None, system_instruction=None, **kwargs):
        self.prompts.append(self._flatten(input))
        return ToolCallResponse(
            tool_calls=[ToolCall(name=n, arguments=a) for n, a in self.tool_plan],
            content="stub tool routing",
        )

    async def ainvoke_with_tools(self, input, tools, message_history=None, system_instruction=None, **kwargs):
        return self.invoke_with_tools(input, tools, message_history, system_instruction, **kwargs)


print("EchoLLM ready — LLMBase subclass, no network")

EchoLLM ready — LLMBase subclass, no network


#### `Text2CypherRetriever`

No embeddings and no index. It hands the LLM the database schema and asks for a Cypher query, then runs it.
The schema is the whole ballgame: everything the model knows about your graph comes from that string.

In [32]:
from neo4j_graphrag.schema import get_schema

schema = get_schema(driver, database=config.database)
print(f"schema string: {len(schema)} characters\n")
print(schema[:700], "...")

schema string: 4447 characters

Node properties:
BusinessEvent {canon_id: STRING, embedding: LIST, name: STRING, type: STRING, n_docs: INTEGER, aliases: LIST, docs: LIST, n_mentions: INTEGER, modality: STRING}
BusinessSegment {canon_id: STRING, type: STRING, name: STRING, embedding: LIST, n_docs: INTEGER, aliases: LIST, docs: LIST, n_mentions: INTEGER}
Commodity {canon_id: STRING, type: STRING, name: STRING, embedding: LIST, n_docs: INTEGER, aliases: LIST, docs: LIST, n_mentions: INTEGER}
Company {canon_id: STRING, type: STRING, name: STRING, embedding: LIST, n_docs: INTEGER, aliases: LIST, docs: LIST, n_mentions: INTEGER}
FinancialMetric {canon_id: STRING, type: STRING, name: STRING, embedding: LIST, n_docs: INTEGER, alia ...


Note `embedding: LIST` in there. `get_schema` reports every property, so the 384-float vectors we added in §4
are advertised to the model as things it might usefully `RETURN`. On a real graph with several embedded
labels this is a meaningful slice of the prompt spent on noise. Trim it.

In [33]:
import re

# Drop embedding properties from the schema handed to the LLM.
lean_schema = re.sub(r"\s*embedding: LIST[^,)\n]*,?", "", schema)
print(f"{len(schema)} -> {len(lean_schema)} characters "
      f"({(1 - len(lean_schema)/len(schema)):.0%} smaller)")

text2cypher = Text2CypherRetriever(
    driver,
    llm=EchoLLM(canned=
        "```cypher\n"
        "MATCH (c:Company)-[:OPERATES_IN]->(g:Geography)\n"
        "RETURN c.name AS company, collect(g.name) AS geographies\n"
        "ORDER BY size(geographies) DESC LIMIT 5\n"
        "```"),
    neo4j_schema=lean_schema,
    examples=["USER INPUT: 'Which companies operate in Malaysia?' "
              "QUERY: MATCH (c:Company)-[:OPERATES_IN]->(g:Geography) "
              "WHERE g.name CONTAINS 'Malaysia' RETURN c.name"],
    neo4j_database=config.database,
)

result = text2cypher.search(query_text="Where does each company operate?")
print("\ngenerated Cypher (markdown fences stripped automatically):")
print(" ", result.metadata["cypher"].replace("\n", "\n  "))
print(f"\n{len(result.items)} rows:")
for item in result.items:
    print("  ", str(item.content)[:130])

4447 -> 4226 characters (5% smaller)

generated Cypher (markdown fences stripped automatically):
  MATCH (c:Company)-[:OPERATES_IN]->(g:Geography)
  RETURN c.name AS company, collect(g.name) AS geographies
  ORDER BY size(geographies) DESC LIMIT 5
  

4 rows:
   <Record company='Northwind Logistics Inc' geographies=['Alberta', 'United States', 'Arizona', 'Canada', 'European Union', 'Texas']
   <Record company='Halcyon Semiconductor Corporation' geographies=['PHOENIX', 'Penang Malaysia', 'Dresden Germany', 'Phoenix Arizona
   <Record company='Vantage Energy Partners' geographies=['Arizona', 'Alberta', 'Texas']>
   <Record company='Cascade Freight Systems' geographies=['Pacific Northwest']>


The safety rail is worth seeing, because it is the obvious thing to worry about when an LLM writes queries against your database:

In [34]:
from neo4j_graphrag.exceptions import Text2CypherRetrievalError

hostile = Text2CypherRetriever(
    driver,
    llm=EchoLLM(canned="MATCH (c:Company) SET c.compromised = true RETURN c.name"),
    neo4j_schema=lean_schema,
    neo4j_database=config.database,
)
try:
    hostile.search(query_text="quietly modify everything")
    print("write executed — that would be bad")
except Text2CypherRetrievalError as exc:
    print(f"refused: {exc}")

records, _, _ = driver.execute_query(
    "MATCH (c:Company) WHERE c.compromised IS NOT NULL RETURN count(c) AS n",
    routing_=neo4j.RoutingControl.READ)
print(f"nodes modified: {records[0]['n']}")

refused: Refusing to execute non-read-only Cypher (query_type='rw'): MATCH (c:Company) SET c.compromised = true RETURN c.name
nodes modified: 0


The query is classified before execution and anything that is not read-only is refused. Worth knowing the
boundary of that guarantee: it stops *writes*, not *expensive reads*. A generated cartesian product will
happily run. `Text2CypherRetriever` also has **no `top_k`** — if you want a bound, it goes in your examples
or your custom prompt.

#### `ToolsRetriever` — routing, not retrieval

In [35]:
doc_tool = doc_vector.convert_to_tool(
    name="document_search",
    description="Semantic search over the source news documents. Use for 'what happened' questions.",
    parameter_descriptions={"query_text": "natural-language query", "top_k": "how many documents"},
)
entity_tool = entity_hybrid.convert_to_tool(
    name="entity_lookup",
    description="Find a specific company, person or metric by name, alias or ticker, with its neighbourhood.",
    parameter_descriptions={"query_text": "the entity name or description", "top_k": "how many entities"},
)

router = ToolsRetriever(
    driver,
    llm=EchoLLM(tool_plan=[("entity_lookup", {"query_text": "NWL", "top_k": 2})]),
    tools=[doc_tool, entity_tool],
    neo4j_database=config.database,
)

result = router.search(query_text="What do we know about NWL?")
print("tools offered :", [t.get_name() for t in [doc_tool, entity_tool]])
print("tool selected :", result.metadata.get("tools_selected"))
print("\nresult:")
for item in result.items:
    print("  " + str(item.content).replace("\n", "\n  ")[:300])
print("\nNote the raw <Record ...> shape: ToolsRetriever calls the underlying retriever's")
print("search() with the LLM's arguments, and does not apply its result_formatter.")
print("Note the ranking too: the scripted call sends only query_text and top_k, so the")
print("lookup ran with the default naive ranker -- the §5.5 tie, Northwind level with")
print("Dresden at 1.000 and the order decided by nothing.")
print("\nranker/alpha in the generated tool schema:",
      sorted(set(entity_tool.get_parameters()["properties"]) & {"ranker", "alpha"}))

tools offered : ['document_search', 'entity_lookup']
tool selected : ['entity_lookup']

result:
  <Record content='(company) Northwind Logistics Inc  (aka NWL, Northwind, Northwind Logistics, Northwind Logistics Inc.)\n      OPERATES_IN -> Alberta\n      PARTICIPANT_IN -> agreement\n      OPERATES_IN -> United States\n      REPORTS_METRIC -> operating margin\n      REPORTS_METRIC -> margin\n    
  <Record content='(geography) Dresden\n      ' tool_name='entity_lookup' metadata={'name': 'Dresden', 'tool': 'entity_lookup'}>

Note the raw <Record ...> shape: ToolsRetriever calls the underlying retriever's
search() with the LLM's arguments, and does not apply its result_formatter.
Note the ranking too: the scripted call sends only query_text and top_k, so the
lookup ran with the default naive ranker -- the §5.5 tie, Northwind level with
Dresden at 1.000 and the order decided by nothing.

ranker/alpha in the generated tool schema: ['alpha', 'ranker']


With a real tool-calling model the `tool_plan` above is what the LLM decides. Look at what it did not send,
though: only `query_text` and `top_k`, so the hybrid search fell back to the default `naive` ranker and
reproduced §5.5's tie — *Dresden*, which has no lexical match to `NWL` at all, level with Northwind. It came
second this time; nothing in the ranking put it there. `ranker` and `alpha` *are* in the tool schema, as the
line above shows: `convert_to_tool` reads them off `get_search_results`, so a model *can* send them, but
nothing makes it. If that choice should not be the LLM's, bind the kwargs — a `functools.partial` or a thin
wrapper — before converting the retriever to a tool.

The stub scripts the call so the wiring is visible: `ToolsRetriever` is a router over the other retrievers,
and it is the right shape when you have several indexes serving genuinely different questions — which, by
§5.5, you do.

#### `GraphRAG` — retrieval plus generation

The last piece assembles retrieved context into a prompt and asks for an answer. Everything above is
retrieval; this is the "G".

In [36]:
from neo4j_graphrag.generation import GraphRAG

QUESTION = "What regulatory trouble is Northwind Logistics in, and who runs the company?"

stub_rag = GraphRAG(retriever=doc_graph, llm=EchoLLM())
answer = stub_rag.search(QUESTION, retriever_config={"top_k": 3}, return_context=True)

print(f"answer (stub): {answer.answer}\n")
print(f"context actually retrieved and sent to the model ({len(answer.retriever_result.items)} items):\n")
for item in answer.retriever_result.items:
    print("  " + str(item.content).replace("\n", "\n  ")[:340] + "\n")

answer (stub): [echo-stub] received 1811 characters of prompt; a real model would answer here.

context actually retrieved and sent to the model (3 items):

  [d03] Northwind Logistics Inc. Form 8-K — Item 5.02 Director Election and Officer Retirement; Item 1A Risk Factor Update
      entities: supply disruptions, Torrent Microsystems, telematics hardware, Elena Vasquez, Freight Brokerage segment, Northwind Logistics Inc, Marcus Webb, margins
      facts:    Elena Vasquez -[OFFICER_OF]-> Northw

  [d02] US regulator opens review of Northwind-Cascade freight deal as Meridian pact expands
      entities: Kansas City, Chicago, Northwind Logistics Inc, trucking, Canada, Cascade brand, European Union, Meridian Rail Group
      facts:    Northwind Logistics Inc -[OPERATES_IN]-> Alberta; Northwind Logistics Inc -[PARTICIPANT_IN]-> agreeme

  [d07] Northwind Logistics Inc. Form 8-K — Item 8.01 Other Events
      entities: Securities and Exchange Commission, Surface Transportation Board, Priya 

The stub's "answer" is a character count, and that is the point — **what matters here is the context**, which
is really retrieved: it ran against the database, bad edges and all, as §5.3 showed. Those three items are
exactly what a live model would receive: three documents, each carrying eight of the entities extracted from
it and five of their relationships. That is what makes this GraphRAG rather than RAG, and the truncation is
what makes it a *slice* of the graph rather than the graph.

`return_context=True` is worth passing explicitly. It is deprecated to omit it (the default is flipping) and
you almost always want to see what was retrieved when an answer is wrong.

In [37]:
if live_llm is not None:
    live_rag = GraphRAG(retriever=doc_graph, llm=live_llm)
    live = live_rag.search(QUESTION, retriever_config={"top_k": 3}, return_context=True)
    print("── live answer ──")
    print(live.answer)
else:
    print("Skipped — no ANTHROPIC_API_KEY set.")
    print("The context above is what would have been sent; only the generation step is missing.")

Skipped — no ANTHROPIC_API_KEY set.
The context above is what would have been sent; only the generation step is missing.


---

## 6. Visualization with NVL

[`neo4j-viz`](https://neo4j.com/docs/neo4j-viz/current/) wraps the Neo4j Visualization Library — the same
rendering engine behind Bloom and the Aura Explore tab — as a Python package that draws inside a notebook.

Two output modes, and the choice matters:

- **`render()`** returns an `IPython.display.HTML` object with the whole NVL bundle inlined. It works
  anywhere HTML works, including a statically-exported notebook. It is also **~8.5 MB per call**.
- **`render_widget()`** returns an `anywidget`, which is far smaller but needs a live kernel — it renders
  nothing in a static export or on GitHub.

This notebook uses `render()` and keeps the count low, because a visualization nobody can see in the
committed file is not much of a visualization.

In [38]:
from neo4j_viz import VisualizationGraph, Node, Relationship
from neo4j_viz.neo4j import from_neo4j

# The entity subgraph only. MENTIONED_IN is 173 of 241 relationships -- essential for
# retrieval, and it would turn any drawing into a hairball centred on ten documents.
graph_result = driver.execute_query(
    """
    MATCH p = (a:__Entity__)-[r]->(b:__Entity__)
    WHERE type(r) <> 'MENTIONED_IN'
    RETURN p
    """,
    routing_=neo4j.RoutingControl.READ,
    result_transformer_=neo4j.Result.graph,
)

vg = from_neo4j(graph_result)
print(f"{len(vg.nodes)} nodes, {len(vg.relationships)} relationships")
print(f"default caption: {vg.nodes[0].caption!r}")
print(f"node properties: {sorted(vg.nodes[0].properties)}")

53 nodes, 68 relationships
default caption: 'Person:__Entity__'
node properties: ['aliases', 'canon_id', 'docs', 'embedding', 'labels', 'n_docs', 'n_mentions', 'name', 'type']


Two things to fix before drawing.

The default caption is every label joined — `'Person:__Entity__'` above — because `from_neo4j` concatenates
them, and the shared label we added in §4 for indexing now shows up on every node. And `properties` carries
`embedding`: 384 floats per node, copied verbatim into the rendered HTML. A map projection in the Cypher does
not help, because `from_neo4j` reads the hydrated graph rather than the returned columns. Strip it in Python.

In [39]:
# The ontology has 13 entity types and neo4j-viz ships 12 default colours, so it
# silently reuses one. Supply the palette explicitly -- the same one notebook 01
# uses for its matplotlib renders, so the two agree.
from kgx.graph import PALETTE

def prepare(vg, *, drop=("embedding",), caption="name", color_by="type", size_by="n_docs"):
    """Strip bulky properties and apply consistent styling."""
    for node in vg.nodes:
        for key in drop:
            node.properties.pop(key, None)
    vg.set_node_captions(property=caption)
    vg.color_nodes(property=color_by, colors=PALETTE)
    vg.resize_nodes(property=size_by, node_radius_min_max=(8, 45))
    return vg

before = sum(len(n.properties.get("embedding", [])) for n in vg.nodes)
prepare(vg)
print(f"dropped {before:,} embedding floats from the render payload")
print(f"caption now: {vg.nodes[0].caption!r}   color: {vg.nodes[0].color}   size: {vg.nodes[0].size}")

dropped 20,352 embedding floats from the render payload
caption now: 'Thomas Ingersoll'   color: #4c78a8   size: 8.0


`color_nodes(property="type")` uses the ontology type we stored on every node, which is why the palette lines
up with the node labels rather than with the joined-label strings. `resize_nodes(property="n_docs")`
makes entities that appear across more documents physically larger — corroboration made visible.

In [40]:
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

# Not `show` -- that name belongs to §5's RetrieverResult printer, and rebinding it
# breaks every retriever cell above for anyone who re-runs one after reaching §6.
def render_and_save(vg, name, **kwargs):
    """Render inline and save a standalone copy."""
    html = vg.render(**kwargs)
    (OUT / f"{name}.html").write_text(html.data)
    print(f"saved {OUT.name}/{name}.html  ({len(html.data)/1e6:.1f} MB)")
    return html

render_and_save(vg, "nvl_entity_graph", layout="forcedirected", height="620px")

saved output/nvl_entity_graph.html  (8.5 MB)


[NVL interactive render — 8.5 MB of inlined bundle, stripped from the committed file]
Re-run this cell for the live version, or open the standalone copy
it wrote to output/ in a browser.


Drag the nodes; hover for properties. The `Northwind` hub is visibly the centre of this corpus, and the
relationship captions are the ontology's relation types.

### Visualizing what a retriever actually returned

The more useful application is not "draw the whole graph" — it is **draw what the retriever reached**. Below
is the neighbourhood the `VectorCypherRetriever` drew on for the §5.6 question: the top-3 documents, every
entity extracted from them, and how those entities relate. The prompt itself got a slice of this, not all of
it — `DOC_TO_GRAPH` keeps `[..8]` entity names per document and `format_doc_context` keeps `[:5]` facts, so
at most 24 names and exactly 15 facts of what is drawn here reached the model.

This is the debugging view for a RAG pipeline. When an answer is wrong, the question is almost always *what
did it actually retrieve* — and that is a subgraph, not a list of strings.

In [41]:
QUESTION = "What regulatory trouble is Northwind Logistics in, and who runs the company?"
top_docs = [item.metadata["doc_id"] for item in doc_graph.search(query_text=QUESTION, top_k=3).items]
print(f"retrieved documents: {top_docs}\n")

context_result = driver.execute_query(
    """
    MATCH (d:Document) WHERE d.doc_id IN $docs
    MATCH p = (e:__Entity__)-[:MENTIONED_IN]->(d)
    WITH collect(p) AS paths, collect(DISTINCT e) AS entities
    UNWIND entities AS a
    OPTIONAL MATCH q = (a)-[r]->(b:__Entity__) WHERE b IN entities AND type(r) <> 'MENTIONED_IN'
    RETURN paths, collect(q) AS facts
    """,
    docs=top_docs, routing_=neo4j.RoutingControl.READ,
    result_transformer_=neo4j.Result.graph,
)

ctx = from_neo4j(context_result)
for node in ctx.nodes:
    node.properties.pop("embedding", None)
    node.properties.pop("text", None)          # document bodies are large and unreadable in a tooltip

# Documents have no `name`; fall back so every node is captioned.
for node in ctx.nodes:
    node.caption = node.properties.get("name") or node.properties.get("doc_id") or node.caption
ctx.color_nodes(property="type", colors=PALETTE)
ctx.resize_nodes(property="n_docs", node_radius_min_max=(10, 40))

print(f"retrieved context as a graph: {len(ctx.nodes)} nodes, {len(ctx.relationships)} relationships")
render_and_save(ctx, "nvl_retrieved_context", layout="forcedirected", height="620px")

retrieved documents: ['d03', 'd02', 'd07']



retrieved context as a graph: 40 nodes, 64 relationships
saved output/nvl_retrieved_context.html  (8.5 MB)


[NVL interactive render — 8.5 MB of inlined bundle, stripped from the committed file]
Re-run this cell for the live version, or open the standalone copy
it wrote to output/ in a browser.


That picture is the neighbourhood the context window was cut from: three documents, the entities they
mention, and the relationships among those entities. §5.6's prompt carried a truncated slice of it — the two
caps in §5.3 decide which slice, and neither of them looks at `support`. The *shape* is what distinguishes
GraphRAG from chunk retrieval either way. A plain vector RAG pipeline would have sent three blocks of prose
and none of the edges.

> **A note on file size.** Each `render()` call inlines the full NVL bundle, so two renders add ~17 MB to
> this notebook. Both are also written to `output/*.html` as standalone pages you can open directly. If you
> are committing this to a repository, consider clearing those two outputs (`Cell → Current Outputs → Clear`)
> and relying on the exported HTML.
>
> One more wrinkle: after `jupyter nbconvert --execute`, the notebook is unsigned and JupyterLab strips the
> inline scripts, so the renders appear blank until you run `jupyter trust notebooks/02_*.ipynb`.

---

## 7. What this bought

The graph went from a Python object to a queryable database, and three quite different access patterns now
work over the same data:

| | good at | blind to |
|---|---|---|
| **Cypher** | exact structure, multi-hop paths, aggregation, provenance filtering | anything phrased in natural language |
| **Retrievers** | natural-language entry points, paraphrase, aliases | multi-hop reasoning; they find a *starting node* |
| **`VectorCypherRetriever`** | both — semantic entry, then structural expansion | still bounded by the traversal you wrote |

The middle row is the honest limitation of vector RAG, and the third row is the whole argument for putting a
graph underneath it. Semantic search finds *where to start*; the graph decides *what is relevant from there*.
Neither half does the other's job.

And the pipeline is now end to end: **text → GLiNER2.5 → ontology-typed graph → entity resolution → Neo4j →
retrieval → generation**, with every edge still carrying the sentence it came from.

In [42]:
summary = pd.DataFrame([
    {"stage": "source documents",        "count": len(DOCUMENTS)},
    {"stage": "canonical entities",      "count": counts(driver)["by_label"].get("__Entity__", 0)},
    {"stage": "relationships (semantic)","count": sum(v for k, v in counts(driver)["by_type"].items()
                                                      if k != "MENTIONED_IN")},
    {"stage": "document links",          "count": counts(driver)["by_type"].get("MENTIONED_IN", 0)},
    {"stage": "vector + fulltext indexes","count": len(INDEXES)},
    {"stage": "retrievers demonstrated", "count": 6},
]).set_index("stage")
display(summary)

print(f"browse it at {config.uri.replace('bolt://', 'http://').replace('7690', '7476')}")
print(f"standalone visualizations: {', '.join(p.name for p in sorted(OUT.glob('nvl_*.html')))}")

,count
stage,
source documents,10
canonical entities,109
relationships (semantic),68
document links,173
vector + fulltext indexes,4
retrievers demonstrated,6


browse it at http://localhost:7476
standalone visualizations: nvl_entity_graph.html, nvl_retrieved_context.html


## Where to take it

**Retrieval quality.** Nothing here is evaluated. Build a small question set with known answers and measure
recall@k for each retriever — the ranker and `alpha` choices in §5.4 were argued from a handful of examples,
which is exactly the standard this notebook criticises elsewhere.

**Chunking.** Documents are embedded whole. Real corpora need chunking, which introduces its own
questions — chunk-to-entity linking, parent-document retrieval, and whether a chunk or a document is the
right retrieval unit. `neo4j-graphrag` ships a `SimpleKGPipeline` that does LLM-based extraction *and*
chunking; comparing its output to the GLiNER graph on the same corpus is the natural next experiment.

**The agent-memory graph.** Only the business-news graph was loaded. The memory graph needs the temporal
layer from notebook 01 to survive the trip — validity intervals as relationship properties, and every query
filtered to current facts. That is a different and more interesting loading problem.

**Write-back.** Retrieval is read-only here. An agent that *learns* writes new episodes back, which means
running resolution against the live graph rather than a batch — `kgx.CanonicalRegistry` against Neo4j instead
of an in-memory dict.

**Scale.** 109 entities fits in a browser and in a prompt. At 10⁶ the interesting problems are all different:
index maintenance, embedding refresh on re-extraction, subgraph sampling for visualization, and retrieval
that has to be selective rather than exhaustive.

In [43]:
driver.close()
print("driver closed")

driver closed
